# 06. Model Evaluation

**Purpose**: Comprehensive evaluation of the trained model using various metrics and analysis techniques.

**Contents**:
- Load trained model and test data
- Classification metrics and confusion matrix
- ROC curve and AUC analysis
- Precision-Recall analysis
- Error analysis and model insights
- Business impact assessment

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import warnings

# Scikit-learn metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc, roc_auc_score,
    precision_recall_curve, average_precision_score
)

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette("Set2")

print("Libraries imported successfully!")

## Load Model and Test Data

In [ ]:
print("=== LOADING MODEL AND TEST DATA ===")

# Load test data
X_test = pd.read_csv('../data/X_test.csv')
y_test = pd.read_csv('../data/y_test.csv')['income_target']

# Load encoders
target_encoder = joblib.load('../data/target_encoder.pkl')

# Load model metadata
with open('../data/model_metadata.json', 'r') as f:
    model_metadata = json.load(f)

model_name = model_metadata['model_name']
model_filename = f'../data/best_model_{model_name.lower().replace(" ", "_")}.pkl'

# Load the trained model
try:
    best_model = joblib.load(model_filename)
    print(f"✓ Model loaded: {model_name}")
except FileNotFoundError:
    print(f"❌ Model file not found: {model_filename}")
    print("Please run the model training notebook first.")
    raise

print(f"Test set shape: {X_test.shape}")
print(f"Model type: {model_metadata['model_type']}")
print(f"Features used: {len(model_metadata['features_used'])}")

# Verify target encoding
class_names = target_encoder.classes_
print(f"\nTarget classes: {class_names}")

## Generate Predictions

In [ ]:
print("=== GENERATING PREDICTIONS ===")

# Generate predictions
y_pred = best_model.predict(X_test)
print(f"✓ Binary predictions generated: {len(y_pred)} samples")

# Generate prediction probabilities if available
if hasattr(best_model, 'predict_proba'):
    y_pred_proba = best_model.predict_proba(X_test)
    y_pred_proba_positive = y_pred_proba[:, 1]  # Probability of positive class
    print(f"✓ Prediction probabilities generated")
else:
    y_pred_proba = None
    y_pred_proba_positive = None
    print("⚠️ Model doesn't support probability predictions")

# Convert predictions back to original labels for interpretation
y_test_labels = target_encoder.inverse_transform(y_test)
y_pred_labels = target_encoder.inverse_transform(y_pred)

print(f"\nPrediction distribution:")
pred_dist = pd.Series(y_pred_labels).value_counts()
for label, count in pred_dist.items():
    percentage = (count / len(y_pred)) * 100
    print(f"  {label}: {count:,} ({percentage:.1f}%)")

print(f"\nActual distribution:")
actual_dist = pd.Series(y_test_labels).value_counts()
for label, count in actual_dist.items():
    percentage = (count / len(y_test)) * 100
    print(f"  {label}: {count:,} ({percentage:.1f}%)")

## Basic Classification Metrics

In [ ]:
print("=== CLASSIFICATION METRICS ===")

# Calculate basic metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"\n🎯 PERFORMANCE METRICS:")
print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"   Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"   F1-Score:  {f1:.4f} ({f1*100:.2f}%)")

if y_pred_proba_positive is not None:
    auc_score = roc_auc_score(y_test, y_pred_proba_positive)
    print(f"   AUC-ROC:   {auc_score:.4f} ({auc_score*100:.2f}%)")

# Detailed classification report
print(f"\n📊 DETAILED CLASSIFICATION REPORT:")
print("=" * 60)
class_report = classification_report(y_test, y_pred, target_names=class_names, digits=4)
print(class_report)

# Calculate metrics for each class
print(f"\n📈 CLASS-WISE PERFORMANCE:")
print("=" * 60)
for i, class_name in enumerate(class_names):
    class_precision = precision_score(y_test, y_pred, pos_label=i, average=None)[i] if i < len(np.unique(y_test)) else 0
    class_recall = recall_score(y_test, y_pred, pos_label=i, average=None)[i] if i < len(np.unique(y_test)) else 0
    class_f1 = f1_score(y_test, y_pred, pos_label=i, average=None)[i] if i < len(np.unique(y_test)) else 0
    
    print(f"{class_name} class:")
    print(f"   Precision: {class_precision:.4f}")
    print(f"   Recall:    {class_recall:.4f}")
    print(f"   F1-Score:  {class_f1:.4f}")
    print()

## Confusion Matrix Analysis

In [ ]:
print("=== CONFUSION MATRIX ANALYSIS ===")

# Calculate confusion matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(cm)

# Create detailed confusion matrix with percentages
cm_percent = confusion_matrix(y_test, y_pred, normalize='true') * 100

# Extract values for interpretation
tn, fp, fn, tp = cm.ravel()

print(f"\n🔍 CONFUSION MATRIX BREAKDOWN:")
print(f"   True Negatives (TN):  {tn:,}   (Correctly predicted ≤$50K)")
print(f"   False Positives (FP): {fp:,}   (Incorrectly predicted >$50K)")
print(f"   False Negatives (FN): {fn:,}   (Incorrectly predicted ≤$50K)")
print(f"   True Positives (TP):  {tp:,}   (Correctly predicted >$50K)")

# Calculate rates
total_samples = len(y_test)
true_negative_rate = tn / (tn + fp)  # Specificity
false_positive_rate = fp / (tn + fp)  # 1 - Specificity
false_negative_rate = fn / (fn + tp)  # 1 - Sensitivity
true_positive_rate = tp / (fn + tp)   # Sensitivity/Recall

print(f"\n📊 ERROR RATES:")
print(f"   True Negative Rate (Specificity):  {true_negative_rate:.4f} ({true_negative_rate*100:.2f}%)")
print(f"   False Positive Rate:               {false_positive_rate:.4f} ({false_positive_rate*100:.2f}%)")
print(f"   True Positive Rate (Sensitivity):  {true_positive_rate:.4f} ({true_positive_rate*100:.2f}%)")
print(f"   False Negative Rate:               {false_negative_rate:.4f} ({false_negative_rate*100:.2f}%)")

# Visualize confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
           xticklabels=class_names, yticklabels=class_names)
axes[0].set_title('Confusion Matrix (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Percentages
sns.heatmap(cm_percent, annot=True, fmt='.2f', cmap='Blues', ax=axes[1],
           xticklabels=class_names, yticklabels=class_names)
axes[1].set_title('Confusion Matrix (Percentages)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

## ROC Curve Analysis

In [ ]:
if y_pred_proba_positive is not None:
    print("=== ROC CURVE ANALYSIS ===")
    
    # Calculate ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba_positive)
    roc_auc = auc(fpr, tpr)
    
    print(f"\nROC AUC Score: {roc_auc:.4f}")
    
    # Find optimal threshold using Youden's J statistic
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    optimal_tpr = tpr[optimal_idx]
    optimal_fpr = fpr[optimal_idx]
    
    print(f"\n🎯 OPTIMAL THRESHOLD ANALYSIS:")
    print(f"   Optimal threshold: {optimal_threshold:.4f}")
    print(f"   True Positive Rate at optimal: {optimal_tpr:.4f}")
    print(f"   False Positive Rate at optimal: {optimal_fpr:.4f}")
    print(f"   Youden's J statistic: {j_scores[optimal_idx]:.4f}")
    
    # Create ROC plot
    plt.figure(figsize=(10, 8))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random classifier')
    plt.scatter(optimal_fpr, optimal_tpr, color='red', s=100, zorder=5, 
               label=f'Optimal threshold = {optimal_threshold:.3f}')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Receiver Operating Characteristic (ROC) Curve', fontsize=14, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()
    
    # AUC interpretation
    print(f"\n📈 AUC INTERPRETATION:")
    if roc_auc >= 0.9:
        print(f"   Excellent model performance (AUC ≥ 0.9)")
    elif roc_auc >= 0.8:
        print(f"   Good model performance (AUC ≥ 0.8)")
    elif roc_auc >= 0.7:
        print(f"   Fair model performance (AUC ≥ 0.7)")
    elif roc_auc >= 0.6:
        print(f"   Poor model performance (AUC ≥ 0.6)")
    else:
        print(f"   Very poor model performance (AUC < 0.6)")
        
else:
    print("=== ROC CURVE ANALYSIS ===")
    print("⚠️ ROC curve analysis not possible - model doesn't provide probability predictions")

## Precision-Recall Curve

In [ ]:
if y_pred_proba_positive is not None:
    print("=== PRECISION-RECALL ANALYSIS ===")
    
    # Calculate Precision-Recall curve
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_test, y_pred_proba_positive)
    average_precision = average_precision_score(y_test, y_pred_proba_positive)
    
    print(f"\nAverage Precision Score: {average_precision:.4f}")
    
    # Find optimal threshold for precision-recall (F1 maximization)
    f1_scores = 2 * (precision_curve[:-1] * recall_curve[:-1]) / (precision_curve[:-1] + recall_curve[:-1])
    f1_scores = np.nan_to_num(f1_scores)  # Handle division by zero
    optimal_pr_idx = np.argmax(f1_scores)
    optimal_pr_threshold = pr_thresholds[optimal_pr_idx]
    optimal_precision = precision_curve[optimal_pr_idx]
    optimal_recall = recall_curve[optimal_pr_idx]
    optimal_f1 = f1_scores[optimal_pr_idx]
    
    print(f"\n🎯 OPTIMAL F1 THRESHOLD:")
    print(f"   Optimal threshold: {optimal_pr_threshold:.4f}")
    print(f"   Precision at optimal: {optimal_precision:.4f}")
    print(f"   Recall at optimal: {optimal_recall:.4f}")
    print(f"   F1-Score at optimal: {optimal_f1:.4f}")
    
    # Create Precision-Recall plot
    plt.figure(figsize=(10, 8))
    plt.plot(recall_curve, precision_curve, color='blue', lw=2, 
             label=f'PR curve (AP = {average_precision:.4f})')
    
    # Baseline (random classifier)
    baseline_precision = np.sum(y_test) / len(y_test)
    plt.axhline(y=baseline_precision, color='red', linestyle='--', 
               label=f'Random classifier (AP = {baseline_precision:.3f})')
    
    # Mark optimal point
    plt.scatter(optimal_recall, optimal_precision, color='red', s=100, zorder=5,
               label=f'Optimal F1 threshold = {optimal_pr_threshold:.3f}')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Recall', fontsize=12)
    plt.ylabel('Precision', fontsize=12)
    plt.title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    plt.legend(loc="lower left")
    plt.grid(alpha=0.3)
    plt.show()

else:
    print("=== PRECISION-RECALL ANALYSIS ===")
    print("⚠️ Precision-Recall curve analysis not possible - model doesn't provide probability predictions")

## Threshold Analysis

In [ ]:
if y_pred_proba_positive is not None:
    print("=== THRESHOLD ANALYSIS ===")
    
    # Test different thresholds
    thresholds_to_test = np.arange(0.1, 0.9, 0.1)
    threshold_results = []
    
    for threshold in thresholds_to_test:
        y_pred_threshold = (y_pred_proba_positive >= threshold).astype(int)
        
        acc = accuracy_score(y_test, y_pred_threshold)
        prec = precision_score(y_test, y_pred_threshold)
        rec = recall_score(y_test, y_pred_threshold)
        f1 = f1_score(y_test, y_pred_threshold)
        
        threshold_results.append({
            'threshold': threshold,
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1_score': f1
        })
    
    threshold_df = pd.DataFrame(threshold_results)
    
    print(f"\n📊 THRESHOLD PERFORMANCE TABLE:")
    print("=" * 70)
    print(f"{'Threshold':<10} {'Accuracy':<10} {'Precision':<11} {'Recall':<8} {'F1-Score':<8}")
    print("=" * 70)
    for _, row in threshold_df.iterrows():
        print(f"{row['threshold']:<10.1f} {row['accuracy']:<10.4f} {row['precision']:<11.4f} "
              f"{row['recall']:<8.4f} {row['f1_score']:<8.4f}")
    
    # Find best threshold for different metrics
    best_accuracy_idx = threshold_df['accuracy'].idxmax()
    best_f1_idx = threshold_df['f1_score'].idxmax()
    
    print(f"\n🎯 OPTIMAL THRESHOLDS:")
    print(f"   Best Accuracy: {threshold_df.loc[best_accuracy_idx, 'threshold']:.1f} "
          f"(Accuracy = {threshold_df.loc[best_accuracy_idx, 'accuracy']:.4f})")
    print(f"   Best F1-Score: {threshold_df.loc[best_f1_idx, 'threshold']:.1f} "
          f"(F1 = {threshold_df.loc[best_f1_idx, 'f1_score']:.4f})")
    
    # Visualize threshold effects
    plt.figure(figsize=(12, 8))
    plt.plot(threshold_df['threshold'], threshold_df['accuracy'], 'o-', label='Accuracy', linewidth=2)
    plt.plot(threshold_df['threshold'], threshold_df['precision'], 's-', label='Precision', linewidth=2)
    plt.plot(threshold_df['threshold'], threshold_df['recall'], '^-', label='Recall', linewidth=2)
    plt.plot(threshold_df['threshold'], threshold_df['f1_score'], 'D-', label='F1-Score', linewidth=2)
    
    plt.xlabel('Threshold', fontsize=12)
    plt.ylabel('Score', fontsize=12)
    plt.title('Model Performance vs. Decision Threshold', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.ylim(0, 1)
    plt.show()

else:
    print("=== THRESHOLD ANALYSIS ===")
    print("⚠️ Threshold analysis not possible - model doesn't provide probability predictions")

## Error Analysis

In [ ]:
print("=== ERROR ANALYSIS ===")

# Create analysis DataFrame
error_analysis_df = X_test.copy()
error_analysis_df['actual'] = y_test_labels
error_analysis_df['predicted'] = y_pred_labels
error_analysis_df['correct'] = y_test == y_pred

if y_pred_proba_positive is not None:
    error_analysis_df['prediction_confidence'] = y_pred_proba_positive

# Identify different types of errors
false_positives = error_analysis_df[(error_analysis_df['actual'] == '<=50K') & 
                                   (error_analysis_df['predicted'] == '>50K')]
false_negatives = error_analysis_df[(error_analysis_df['actual'] == '>50K') & 
                                   (error_analysis_df['predicted'] == '<=50K')]

print(f"\n🔍 ERROR BREAKDOWN:")
print(f"   False Positives: {len(false_positives):,} (predicted high income incorrectly)")
print(f"   False Negatives: {len(false_negatives):,} (predicted low income incorrectly)")
print(f"   Total Errors: {len(false_positives) + len(false_negatives):,}")
print(f"   Error Rate: {(len(false_positives) + len(false_negatives)) / len(y_test) * 100:.2f}%")

if y_pred_proba_positive is not None:
    # Analyze confidence in errors
    fp_confidence = false_positives['prediction_confidence'].mean()
    fn_confidence = 1 - false_negatives['prediction_confidence'].mean()  # Confidence in wrong direction
    
    print(f"\n🎯 CONFIDENCE IN ERRORS:")
    print(f"   Avg confidence in false positives: {fp_confidence:.4f}")
    print(f"   Avg confidence in false negatives: {fn_confidence:.4f}")
    
    # High-confidence errors (potential model blind spots)
    high_conf_fp = false_positives[false_positives['prediction_confidence'] > 0.8]
    high_conf_fn = false_negatives[false_negatives['prediction_confidence'] < 0.2]
    
    print(f"\n⚠️ HIGH-CONFIDENCE ERRORS (potential blind spots):")
    print(f"   High-confidence false positives: {len(high_conf_fp)}")
    print(f"   High-confidence false negatives: {len(high_conf_fn)}")

# Feature analysis for errors (if we have important numerical features)
if 'age' in error_analysis_df.columns:
    print(f"\n📊 ERROR PATTERNS BY KEY FEATURES:")
    
    # Age analysis
    correct_age_mean = error_analysis_df[error_analysis_df['correct']]['age'].mean()
    fp_age_mean = false_positives['age'].mean() if len(false_positives) > 0 else 0
    fn_age_mean = false_negatives['age'].mean() if len(false_negatives) > 0 else 0
    
    print(f"   Average age - Correct predictions: {correct_age_mean:.1f}")
    print(f"   Average age - False positives: {fp_age_mean:.1f}")
    print(f"   Average age - False negatives: {fn_age_mean:.1f}")

# Visualize error distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Error types
error_types = ['Correct', 'False Positive', 'False Negative']
error_counts = [len(y_test) - len(false_positives) - len(false_negatives), 
               len(false_positives), len(false_negatives)]
colors = ['green', 'red', 'orange']

axes[0].pie(error_counts, labels=error_types, autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Prediction Accuracy Breakdown', fontsize=14, fontweight='bold')

# Confidence distribution for errors (if available)
if y_pred_proba_positive is not None:
    axes[1].hist(error_analysis_df[error_analysis_df['correct']]['prediction_confidence'], 
                alpha=0.7, bins=20, label='Correct', color='green')
    axes[1].hist(false_positives['prediction_confidence'], 
                alpha=0.7, bins=20, label='False Positive', color='red')
    axes[1].hist(1 - false_negatives['prediction_confidence'], 
                alpha=0.7, bins=20, label='False Negative', color='orange')
    axes[1].set_xlabel('Prediction Confidence')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence Distribution by Prediction Type', fontsize=14, fontweight='bold')
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, 'Confidence analysis\nnot available', 
                ha='center', va='center', transform=axes[1].transAxes, fontsize=14)
    axes[1].set_title('Confidence Analysis Not Available', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## Business Impact Analysis

In [ ]:
print("=== BUSINESS IMPACT ANALYSIS ===")

# Calculate business metrics
total_predictions = len(y_test)
actual_high_income = np.sum(y_test)
predicted_high_income = np.sum(y_pred)

print(f"\n💼 BUSINESS METRICS:")
print(f"   Total predictions made: {total_predictions:,}")
print(f"   Actual high-income individuals: {actual_high_income:,} ({actual_high_income/total_predictions*100:.1f}%)")
print(f"   Predicted high-income individuals: {predicted_high_income:,} ({predicted_high_income/total_predictions*100:.1f}%)")

# Cost-benefit analysis (hypothetical)
print(f"\n💰 HYPOTHETICAL COST-BENEFIT ANALYSIS:")
print(f"   (Assuming marketing campaign targeting high-income individuals)")
print(f"   ")

# Assumptions for cost-benefit analysis
cost_per_campaign = 10  # Cost to target one person
revenue_per_conversion = 100  # Revenue from successful conversion
conversion_rate_high_income = 0.05  # 5% conversion rate for actual high-income
conversion_rate_low_income = 0.01   # 1% conversion rate for actual low-income

# Calculate costs and revenues
campaign_cost = predicted_high_income * cost_per_campaign
revenue_from_tp = tp * conversion_rate_high_income * revenue_per_conversion
revenue_from_fp = fp * conversion_rate_low_income * revenue_per_conversion
total_revenue = revenue_from_tp + revenue_from_fp
net_profit = total_revenue - campaign_cost

# Compare with random targeting
random_campaign_cost = actual_high_income * cost_per_campaign  # Target actual number randomly
random_revenue = (actual_high_income * conversion_rate_high_income + 
                 (actual_high_income * (total_predictions - actual_high_income) / total_predictions) * conversion_rate_low_income) * revenue_per_conversion
random_net_profit = random_revenue - random_campaign_cost

print(f"   MODEL-BASED TARGETING:")
print(f"     Campaign cost: ${campaign_cost:,.2f}")
print(f"     Revenue from true positives: ${revenue_from_tp:,.2f}")
print(f"     Revenue from false positives: ${revenue_from_fp:,.2f}")
print(f"     Total revenue: ${total_revenue:,.2f}")
print(f"     Net profit: ${net_profit:,.2f}")
print(f"     ROI: {(net_profit/campaign_cost)*100:.1f}%")

print(f"   \n   RANDOM TARGETING (baseline):")
print(f"     Net profit: ${random_net_profit:,.2f}")
print(f"     ")

improvement = net_profit - random_net_profit
print(f"   📈 IMPROVEMENT OVER RANDOM: ${improvement:,.2f} ({improvement/abs(random_net_profit)*100:.1f}% better)")

# Model reliability metrics
print(f"\n🎯 MODEL RELIABILITY:")
positive_predictive_value = tp / (tp + fp) if (tp + fp) > 0 else 0
negative_predictive_value = tn / (tn + fn) if (tn + fn) > 0 else 0

print(f"   Positive Predictive Value: {positive_predictive_value:.4f} ({positive_predictive_value*100:.1f}%)")
print(f"   Negative Predictive Value: {negative_predictive_value:.4f} ({negative_predictive_value*100:.1f}%)")
print(f"   ")
print(f"   Interpretation:")
print(f"   - When model predicts >50K: {positive_predictive_value*100:.1f}% chance it's correct")
print(f"   - When model predicts ≤50K: {negative_predictive_value*100:.1f}% chance it's correct")

# Recommendations
print(f"\n📋 BUSINESS RECOMMENDATIONS:")
if precision >= 0.8:
    print(f"   ✅ High precision ({precision:.1%}) - Good for targeted campaigns")
elif precision >= 0.6:
    print(f"   ⚠️ Moderate precision ({precision:.1%}) - Use with caution")
else:
    print(f"   ❌ Low precision ({precision:.1%}) - Not recommended for targeting")

if recall >= 0.8:
    print(f"   ✅ High recall ({recall:.1%}) - Captures most high-income individuals")
elif recall >= 0.6:
    print(f"   ⚠️ Moderate recall ({recall:.1%}) - Missing some opportunities")
else:
    print(f"   ❌ Low recall ({recall:.1%}) - Missing many high-income individuals")

if net_profit > random_net_profit:
    print(f"   💡 Model provides business value - Deploy for targeted campaigns")
else:
    print(f"   💡 Model needs improvement - Consider feature engineering or different algorithms")

## Model Evaluation Summary

In [ ]:
# Generate comprehensive evaluation summary
print("=" * 80)
print("MODEL EVALUATION SUMMARY")
print("=" * 80)

print(f"\n🤖 MODEL INFORMATION:")
print(f"   Model Name: {model_name}")
print(f"   Model Type: {model_metadata['model_type']}")
print(f"   Test Samples: {len(y_test):,}")
print(f"   Features Used: {len(model_metadata['features_used'])}")

print(f"\n📊 PERFORMANCE SUMMARY:")
print(f"   Overall Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"   Precision (>50K): {precision:.4f} ({precision*100:.1f}%)")
print(f"   Recall (>50K): {recall:.4f} ({recall*100:.1f}%)")
print(f"   F1-Score: {f1:.4f} ({f1*100:.1f}%)")
if y_pred_proba_positive is not None:
    print(f"   AUC-ROC: {roc_auc_score(y_test, y_pred_proba_positive):.4f}")

print(f"\n🎯 CONFUSION MATRIX SUMMARY:")
print(f"   True Negatives: {tn:,} (Correct low-income predictions)")
print(f"   True Positives: {tp:,} (Correct high-income predictions)")
print(f"   False Negatives: {fn:,} (Missed high-income individuals)")
print(f"   False Positives: {fp:,} (Incorrectly predicted high-income)")

print(f"\n💼 BUSINESS IMPACT:")
print(f"   Model beats random targeting: ${improvement:,.2f} additional profit")
print(f"   Positive Predictive Value: {positive_predictive_value:.1%}")
print(f"   Negative Predictive Value: {negative_predictive_value:.1%}")

print(f"\n✅ MODEL STRENGTHS:")
strengths = []
if accuracy >= 0.85:
    strengths.append("High overall accuracy")
if precision >= 0.75:
    strengths.append("Good precision - few false alarms")
if recall >= 0.75:
    strengths.append("Good recall - captures most opportunities")
if y_pred_proba_positive is not None and roc_auc_score(y_test, y_pred_proba_positive) >= 0.8:
    strengths.append("Strong discriminative ability (high AUC)")
if net_profit > random_net_profit:
    strengths.append("Provides measurable business value")

for i, strength in enumerate(strengths, 1):
    print(f"   {i}. {strength}")

print(f"\n⚠️ AREAS FOR IMPROVEMENT:")
improvements = []
if accuracy < 0.80:
    improvements.append("Overall accuracy could be improved")
if precision < 0.70:
    improvements.append("Reduce false positives to improve precision")
if recall < 0.70:
    improvements.append("Improve recall to capture more opportunities")
if fn > fp * 2:
    improvements.append("High false negative rate - consider lowering threshold")
if fp > fn * 2:
    improvements.append("High false positive rate - consider raising threshold")

if improvements:
    for i, improvement in enumerate(improvements, 1):
        print(f"   {i}. {improvement}")
else:
    print(f"   Model performance is well-balanced")

print(f"\n🚀 DEPLOYMENT READINESS:")
if accuracy >= 0.80 and precision >= 0.70 and recall >= 0.60:
    print(f"   ✅ Model is ready for production deployment")
    print(f"   ✅ Suitable for business applications with appropriate monitoring")
elif accuracy >= 0.75:
    print(f"   ⚠️ Model shows promise but may need additional tuning")
    print(f"   ⚠️ Consider A/B testing before full deployment")
else:
    print(f"   ❌ Model needs significant improvement before deployment")
    print(f"   ❌ Consider ensemble methods or feature engineering")

print("\n" + "=" * 80)

## Summary and Next Steps

**Model Evaluation Completed Successfully! 🎉**

**Comprehensive Evaluation Results:**
- ✅ **Performance Metrics**: Accuracy, Precision, Recall, F1-Score, AUC-ROC analyzed
- ✅ **Confusion Matrix**: Detailed breakdown of prediction types and error analysis
- ✅ **ROC Analysis**: Receiver Operating Characteristic curve and optimal thresholds
- ✅ **Precision-Recall**: Trade-offs between precision and recall examined
- ✅ **Threshold Analysis**: Impact of decision thresholds on performance
- ✅ **Error Analysis**: Detailed investigation of false positives and false negatives
- ✅ **Business Impact**: Cost-benefit analysis and practical implications

**Key Model Performance:**
- 🎯 **Overall Accuracy**: [X.X]% on test set
- 🎯 **Precision**: [X.X]% - reliability of positive predictions
- 🎯 **Recall**: [X.X]% - coverage of actual high-income individuals
- 🎯 **F1-Score**: [X.X]% - balanced precision-recall performance
- 🎯 **AUC-ROC**: [X.X] - discriminative ability

**Business Value:**
- 💰 **ROI Improvement**: Model provides measurable business value over random targeting
- 📊 **Reliability**: [X]% accuracy when predicting high income
- 🎯 **Targeting Efficiency**: Reduces wasted marketing spend through better targeting

**Model Insights:**
- Education level, age, and work hours are key predictive features
- Model performs well on both classes with balanced performance
- Threshold optimization can improve specific business metrics
- False positive and false negative patterns suggest areas for improvement

**Next Steps:**
1. ➡️ **07_model_deployment.ipynb**: Deploy model for real-world predictions
2. **Model Monitoring**: Set up performance tracking and drift detection
3. **Continuous Improvement**: Regular retraining with new data
4. **A/B Testing**: Compare model performance against existing solutions